---
Package Imports
---
---

In [ ]:
from pgfgleam import *

from scipy.linalg import block_diag
from scipy.sparse import csr_array
import pandas as pd
import numpy as np
from scipy.stats import gamma
from scipy import integrate
from scipy import stats
import ast
import re

import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("white")
plt.rcParams['font.family'] = 'sans-serif'

from typing import List
import itertools

import warnings
warnings.filterwarnings('ignore')


In [ ]:
from data_processing import DataSetup

flight_matrix_path = 'pgfgleam/datasets/country_flow_202001 1.csv'
country_codes_path = 'pgfgleam/datasets/geom_countries_codes.csv'
population_path = 'pgfgleam/datasets/WPP2019_TotalPopulation2020.csv'

flight_matrix, normalized_flow_matrix, source_countries, target_countries, source_codes, target_codes, population, _ = DataSetup(flight_matrix_path, population_path, country_codes_path).load_and_process_data().values()


Fixing the GEOM code for DRC

---
Parameter Setup
---
---

In [ ]:
# Epi parameters
R0 = 2
generation_time = 4
# Calculate infectious_period and latency_period from generation time
mean_infectious_period = 8.0 / 3.0  # ~2.67 days
latency_period = generation_time - 0.75 * mean_infectious_period
infectious_period = mean_infectious_period
post_infectious_period = 10

# AWW params
p_det = 0.16
turnaround_time = 3

# Considering SLDR model - where D = detectable is divided into 2 infectious and 2 post-infectious states
nb_latent_states = 1
nb_infectious_states = 2
nb_post_infectious_states = 2
infection='poisson' # can also be nb for negative binomial - then need to define the overdispersion k
nb_microsteps = 1

# for simulations - max time
T_max = 150
t = list(range(0,T_max))

print(f"Doubling time: {calculate_doubling_time(R0,infectious_period,latency_period,nb_latent_states,nb_infectious_states)} days")

---
Matrix Setup
---
---

In [ ]:
# Contact matrix
umat = np.eye(len(normalized_flow_matrix)) * R0 / infectious_period

# Mobility matrix
mmat = normalized_flow_matrix.copy()
np.fill_diagonal(mmat, 0)  # Set diagonal to zero

cmat1 = np.eye(normalized_flow_matrix.shape[0])
cmat2 = np.eye(normalized_flow_matrix.shape[0])

detect_prob = 0.16
smat = np.zeros(normalized_flow_matrix.shape)
sentinel_loc = source_countries.index('United Kingdom')
smat[:, sentinel_loc] = detect_prob
smat[sentinel_loc, sentinel_loc] = 0 # No testing of domestic travel

---
PGF Object
---
---

In [ ]:
# Set origin location
origin_loc = source_countries.index('Ecuador')

# Set false positive rate
fpr = 0.01

# Define states
infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
idx_infected = [origin_loc] 
weight_infected = [1.0]  # Removed age-dependence
emat = np.ones((len(source_countries), len(source_countries)))

Psi = Sentinel2PGF(umat,mmat,smat,emat,latency_period,infectious_period,
                  post_infectious_period,nb_infectious_states,
                  nb_post_infectious_states,infection,nb_microsteps,cmat1=cmat1,cmat2=cmat2,fpr=fpr)


Psi.add_initial_conditions(idx_infected, weight_infected, 
                          label=f'Origin: {source_countries[origin_loc]}', 
                          **infectious_state)

---
Time to First Detection
---
---

In [ ]:
# Set false positive rate
fpr = 0.01

# Max value of time 
T = 90

# Time range over which we want to measure detections
t =  list(range(0,T))

# Transformation that needs to used to measure "cumulative detection" on all possible paths leading to detections
def transform(state_vars,c_):
    state_vars['cumulative detection'] *= c_
    if fpr != 0:
        state_vars['cumulative fp'] *= c_ # Count false positive as well
transform_dict = {max(t):[transform]}  # Measure the cumulative detection for the whole time range, explaining the max(t)

# Time from test to result
turnaround_time = 3 

# Time to 'D' detections: D = 1 is the default we considered
D = 1

# Use quicker marginal_p0 function if D = 1 
if D == 1:
    detections = marginal_p0(Psi, t, transform_dict)

else:
    detections = marginal_distribution(Psi, t, transform_dict)

detections['probability'] = np.real(detections['probability']) #cast to real (the imaginary part is 0 anyway); obtained from FFT
detection_time = get_time_to_detection(detections,D, quantiles=[0.25, 0.5, 0.75, 0.95, 0.99], #time to get D detections
                                    groupbycols=['label'],turnaround_time=turnaround_time)

mean_detection_time = detection_time['mean'].values[0]

Extract Detection Time Statistics

In [ ]:
mean_detection_time = detection_time['mean'].values[0]

median_detection_time = detection_time['0.5'].values[0]

In [ ]:
print(f"Mean detection time: {mean_detection_time} days")

---
Mean Importations
---
---

In [ ]:
# emat can be used to consider origin-level importations (rather than summed across all origins)

emat = np.zeros((len(normalized_flow_matrix), len(normalized_flow_matrix)))
emat[:, sentinel_loc] = 1  # Sentinel location can send to all locations
emat[sentinel_loc, sentinel_loc] = 0  # Sentinel location does not send to itself

K = Sentinel2PGF(umat,mmat,smat,emat,latency_period,infectious_period,
                post_infectious_period,nb_infectious_states,
                nb_post_infectious_states,infection,nb_microsteps,cmat1=cmat1,cmat2=cmat2, cumulant=True)
K.add_initial_conditions(idx_infected, weight_infected,
                        label=f'Origin: {source_countries[origin_loc]}',
                        **infectious_state)

def transform(state_vars, c_):
    state_vars['cumulative importation'] *= c_

def reset_transform(state_vars, c_):
    state_vars['cumulative importation'] = 1  # Reset cumulative importation to zero

transform_dict = {max(t): [transform]}

# Get cumulants data
importations = marginal_cumulants(K, t, transform_dict, radius=10**(-6), resolution=4)

# Consider the cumulative mean number of importations (target = 1)
df = importations[importations['target'] == 1]

# Make df_daily the difference of df cumulant - we can consider the full distribution 
# but this takes a bit longer to run
df_daily = df.copy()
df_daily['cumulant'] = df_daily['cumulant'].diff().fillna(df_daily['cumulant'].iloc[0])

mean_df = pd.DataFrame({
    'time': df['time'],
    'daily_importations': df_daily['cumulant']
})



---
CSVs to Input into Julia Files
---
---

In [ ]:
# daily_imports_sensitivity.csv is a csv for all countries as OLs but summed importations; 
# as this is the full dataset, it does not have the gamma fit parameters 
# but it has the mean distribution for each country-time combination

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================
R0_values = [1.5, 2.0, 2.5, 3.0]
generation_times = [4, 6, 8, 10]
p_det = 0.16
turnaround_time = 3
T_max = 250
t = list(range(0, T_max))

output_filepath = 'daily_imports_sensitivity.csv'

infectious_period_fix = 8.0 / 3.0  # ~2.67 days
n_countries = len(source_countries)
time_arr = np.arange(T_max, dtype=float)

# Pre-compute labels (one per outbreak country)
labels = [f'origin: {source_countries[i]}' for i in range(n_countries)]

# Collect DataFrame chunks
chunks = []

for R0 in R0_values:
    for gen_time in generation_times:

        # --- Latency check ---
        latency_period = gen_time - (0.75 * infectious_period_fix)
        mean_infectious_period = infectious_period_fix

        if latency_period < 0:
            print(f"Skipping R0={R0}, gen_time={gen_time}: negative latency period")
            continue

        print(f"\nR0={R0}, gen_time={gen_time} — batching {n_countries} countries...")

        # --- Build matrices (depend on R0 only, not on outbreak country) ---
        umat_s = np.eye(len(normalized_flow_matrix)) * R0 / mean_infectious_period

        emat = np.zeros([len(smat), len(smat)])
        emat[:, sentinel_loc] = 1
        emat[sentinel_loc, sentinel_loc] = 0

        smat_s = np.zeros([len(mmat), len(mmat)])
        smat_s[:, sentinel_loc] = 0.16
        smat_s[sentinel_loc, sentinel_loc] = 0

        infectious_state = {'nb_infectious': 10, 'nb_latent': 10}

        # ========================================
        # Create Batches PGF objects, each containing all countries as labels
        # ========================================
        K_total = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                               latency_period, infectious_period,
                               post_infectious_period, nb_infectious_states,
                               nb_post_infectious_states, infection, nb_microsteps,
                               cmat1=cmat1, cmat2=cmat2, cumulant=True)

        K_latent = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                                latency_period, infectious_period,
                                post_infectious_period, nb_infectious_states,
                                nb_post_infectious_states, infection, nb_microsteps,
                                cmat1=cmat1, cmat2=cmat2, cumulant=True)

        K_infectious = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                                    latency_period, infectious_period,
                                    post_infectious_period, nb_infectious_states,
                                    nb_post_infectious_states, infection, nb_microsteps,
                                    cmat1=cmat1, cmat2=cmat2, cumulant=True)

        smat_det = np.zeros([len(mmat), len(mmat)])
        smat_det[:, sentinel_loc] = p_det
        smat_det[sentinel_loc, sentinel_loc] = 0

        K_det = SentinelPGF(umat_s, mmat, smat_det, latency_period, infectious_period,
                            post_infectious_period, nb_infectious_states,
                            nb_post_infectious_states, infection, nb_microsteps,
                            cmat1=cmat1, cmat2=cmat2)

        # Add ALL countries as labels to each PGF object
        for outbreak_idx in range(n_countries):
            K_total.add_initial_conditions([outbreak_idx], [1.0],
                                           label=labels[outbreak_idx], **infectious_state)
            K_latent.add_initial_conditions([outbreak_idx], [1.0],
                                            label=labels[outbreak_idx], **infectious_state)
            K_infectious.add_initial_conditions([outbreak_idx], [1.0],
                                                label=labels[outbreak_idx], **infectious_state)
            K_det.add_initial_conditions([outbreak_idx], [1.0],
                                         label=labels[outbreak_idx], **infectious_state)

        # ========================================
        # 3 batched marginal_cumulants calls
        # ========================================

        # Total importations (L + I + P)
        def transform_total(state_vars, c_):
            state_vars['cumulative_latent_importation'] *= c_
            state_vars['cumulative_infectious_importation'] *= c_
            state_vars['cumulative_post_infectious_importation'] *= c_

        # Latent importations (L only)
        def transform_latent(state_vars, c_):
            state_vars['cumulative_latent_importation'] *= c_

        # Infectious importations (I only)
        def transform_infectious(state_vars, c_):
            state_vars['cumulative_infectious_importation'] *= c_

        print("  Running 3 batched marginal_cumulants calls...")
        imp_total = marginal_cumulants(K_total, t,
                                       {max(t): [transform_total]},
                                       radius=1e-6, resolution=2)
        print("    Total (L+I+P) done")

        imp_latent = marginal_cumulants(K_latent, t,
                                        {max(t): [transform_latent]},
                                        radius=1e-6, resolution=2)
        print("    Latent (L) done")

        imp_infectious = marginal_cumulants(K_infectious, t,
                                            {max(t): [transform_infectious]},
                                            radius=1e-6, resolution=2)
        print("    Infectious (I) done")

        # ========================================
        # 1 batched marginal_p0 call for detection
        # ========================================
        def det_transform(state_vars, c_):
            state_vars['cumulative detection'] *= c_

        print("  Running batched detection call...")
        try:
            detections = marginal_p0(K_det, t, {max(t): [det_transform]})
            detections['probability'] = np.real(detections['probability'])
            detection_time_df = get_time_to_detection(
                detections, 1,
                quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                groupbycols=['label'],
                turnaround_time=turnaround_time
            )
            # Build label -> mean detection time mapping
            det_time_map = detection_time_df.groupby('label')['mean'].first().to_dict()
            print("    Detection done")
        except Exception as e:
            print(f"    Detection failed for all countries: {e}, using T_max")
            det_time_map = {}

        # ========================================
        # Extract per-country results
        # ========================================

        # Pre-filter to return mean
        imp_total_t1 = imp_total[imp_total['target'] == 1]
        imp_latent_t1 = imp_latent[imp_latent['target'] == 1]
        imp_infectious_t1 = imp_infectious[imp_infectious['target'] == 1]

        # Group by label for fast lookup
        grp_total = {lbl: grp['cumulant'].values for lbl, grp in imp_total_t1.groupby('label')}
        grp_latent = {lbl: grp['cumulant'].values for lbl, grp in imp_latent_t1.groupby('label')}
        grp_infectious = {lbl: grp['cumulant'].values for lbl, grp in imp_infectious_t1.groupby('label')}

        for outbreak_idx in range(n_countries):
            label = labels[outbreak_idx]
            outbreak_country = source_countries[outbreak_idx]

            # Extract cumulative cumulants for this country
            cum_total = grp_total[label]
            cum_latent = grp_latent[label]
            cum_infectious = grp_infectious[label]

            # Vectorized cumulative-to-daily 
            daily_total = np.empty_like(cum_total)
            daily_total[:-1] = np.diff(cum_total)
            daily_total[-1] = cum_total[-1]

            daily_latent = np.empty_like(cum_latent)
            daily_latent[:-1] = np.diff(cum_latent)
            daily_latent[-1] = cum_latent[-1]

            daily_infectious = np.empty_like(cum_infectious)
            daily_infectious[:-1] = np.diff(cum_infectious)
            daily_infectious[-1] = cum_infectious[-1]

            # Detectable (I+P) = Total (L+I+P) - Latent (L), via cumulant additivity
            daily_detectable = daily_total - daily_latent

            # Detection time for this country
            mean_det_time = det_time_map.get(label, float(T_max))

            # Build chunk DataFrame
            chunk = pd.DataFrame({
                'outbreak_country': outbreak_country,
                'time': time_arr,
                'daily_total_imports': daily_total[:T_max].astype(float),
                'daily_latent_imports': daily_latent[:T_max].astype(float),
                'daily_infectious_imports': daily_infectious[:T_max].astype(float),
                'daily_detectable_imports': daily_detectable[:T_max].astype(float),
                'mean_detection_time': float(mean_det_time),
                'R0': float(R0),
                'generation_time': float(gen_time)
            })
            chunks.append(chunk)

        print(f"  Done — {n_countries} countries processed for R0={R0}, gen_time={gen_time}")

# ============================================================
# Concatenate and save
# ============================================================
daily_df = pd.concat(chunks, ignore_index=True)
daily_df.to_csv(output_filepath, index=False)
print(f"\nComplete! Saved {len(daily_df)} rows to {output_filepath}")


In [ ]:
# daily_imports_full.csv is a csv for all countries as OLs and as importers; 
# as this is the full dataset, it does not have the gamma fit parameters 
# but it has the mean distribution for each country-time combination

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================
R0 = 2.0
gen_time = 4.0
p_det = 0.16
turnaround_time = 3
T_max = 250
t = list(range(0, T_max))

output_filepath = 'daily_imports_full.csv'

infectious_period_fix = 8.0 / 3.0  # ~2.67 days
n_countries = len(source_countries)
time_arr = np.arange(T_max, dtype=float)

# Pre-compute labels (one per outbreak country)
labels = [f'origin: {source_countries[i]}' for i in range(n_countries)]

# Collect DataFrame chunks
chunks = []

for import_country in range(n_countries):

    # --- Latency check ---
    latency_period = gen_time - (0.75 * infectious_period_fix)
    mean_infectious_period = infectious_period_fix

    if latency_period < 0:
        print(f"Skipping R0={R0}, gen_time={gen_time}: negative latency period")
        continue

    print(f"\nR0={R0}, gen_time={gen_time} — batching {n_countries} countries...")

    # --- Build matrices (depend on R0 only, not on outbreak country) ---
    umat_s = np.eye(len(normalized_flow_matrix)) * R0 / mean_infectious_period

    emat = np.zeros([len(smat), len(smat)])
    emat[import_country, sentinel_loc] = 1

    smat_s = np.zeros([len(mmat), len(mmat)])
    smat_s[:, sentinel_loc] = 0.16
    smat_s[sentinel_loc, sentinel_loc] = 0

    infectious_state = {'nb_infectious': 10, 'nb_latent': 10}

    # ========================================
    # Create Batches PGF objects, each containing all countries as labels
    # ========================================
    K_total = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                        latency_period, infectious_period,
                        post_infectious_period, nb_infectious_states,
                        nb_post_infectious_states, infection, nb_microsteps,
                        cmat1=cmat1, cmat2=cmat2, cumulant=True)

    K_latent = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                            latency_period, infectious_period,
                            post_infectious_period, nb_infectious_states,
                            nb_post_infectious_states, infection, nb_microsteps,
                            cmat1=cmat1, cmat2=cmat2, cumulant=True)

    K_infectious = Sentinel4PGF(umat_s, mmat, smat_s, emat,
                                latency_period, infectious_period,
                                post_infectious_period, nb_infectious_states,
                                nb_post_infectious_states, infection, nb_microsteps,
                                cmat1=cmat1, cmat2=cmat2, cumulant=True)

    smat_det = np.zeros([len(mmat), len(mmat)])
    smat_det[:, sentinel_loc] = p_det
    smat_det[sentinel_loc, sentinel_loc] = 0

    # Sentinel3PGF has origin level detections
    K_det = Sentinel3PGF(umat_s, mmat, smat_det, emat, latency_period, infectious_period,
                        post_infectious_period, nb_infectious_states,
                        nb_post_infectious_states, infection, nb_microsteps,
                        cmat1=cmat1, cmat2=cmat2)

    # Add ALL countries as labels to each PGF object
    for outbreak_idx in range(n_countries):
        K_total.add_initial_conditions([outbreak_idx], [1.0],
                                    label=labels[outbreak_idx], **infectious_state)
        K_latent.add_initial_conditions([outbreak_idx], [1.0],
                                        label=labels[outbreak_idx], **infectious_state)
        K_infectious.add_initial_conditions([outbreak_idx], [1.0],
                                            label=labels[outbreak_idx], **infectious_state)
        K_det.add_initial_conditions([outbreak_idx], [1.0],
                                    label=labels[outbreak_idx], **infectious_state)

    # ========================================
    # 3 batched marginal_cumulants calls
    # ========================================

    # Total importations (L + I + P)
    def transform_total(state_vars, c_):
        state_vars['cumulative_latent_importation'] *= c_
        state_vars['cumulative_infectious_importation'] *= c_
        state_vars['cumulative_post_infectious_importation'] *= c_

    # Latent importations (L only)
    def transform_latent(state_vars, c_):
        state_vars['cumulative_latent_importation'] *= c_

    # Infectious importations (I only)
    def transform_infectious(state_vars, c_):
        state_vars['cumulative_infectious_importation'] *= c_

    print("  Running 3 batched marginal_cumulants calls...")
    imp_total = marginal_cumulants(K_total, t,
                                {max(t): [transform_total]},
                                radius=1e-6, resolution=2)
    print("    Total (L+I+P) done")

    imp_latent = marginal_cumulants(K_latent, t,
                                    {max(t): [transform_latent]},
                                    radius=1e-6, resolution=2)
    print("    Latent (L) done")

    imp_infectious = marginal_cumulants(K_infectious, t,
                                        {max(t): [transform_infectious]},
                                        radius=1e-6, resolution=2)
    print("    Infectious (I) done")

    # ========================================
    # 1 batched marginal_p0 call for detection
    # ========================================
    def det_transform(state_vars, c_):
        state_vars['cumulative detection'] *= c_

    print("  Running batched detection call...")
    try:
        detections = marginal_p0(K_det, t, {max(t): [det_transform]})
        detections['probability'] = np.real(detections['probability'])
        detection_time_df = get_time_to_detection(
            detections, 1,
            quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
            groupbycols=['label'],
            turnaround_time=turnaround_time
        )
        # Build label -> mean detection time mapping
        det_time_map = detection_time_df.groupby('label')['mean'].first().to_dict()
        print("    Detection done")
    except Exception as e:
        print(f"    Detection failed for all countries: {e}, using T_max")
        det_time_map = {}

    # ========================================
    # Extract per-country results
    # ========================================

    # Pre-filter to return mean
    imp_total_t1 = imp_total[imp_total['target'] == 1]
    imp_latent_t1 = imp_latent[imp_latent['target'] == 1]
    imp_infectious_t1 = imp_infectious[imp_infectious['target'] == 1]

    # Group by label for fast lookup
    grp_total = {lbl: grp['cumulant'].values for lbl, grp in imp_total_t1.groupby('label')}
    grp_latent = {lbl: grp['cumulant'].values for lbl, grp in imp_latent_t1.groupby('label')}
    grp_infectious = {lbl: grp['cumulant'].values for lbl, grp in imp_infectious_t1.groupby('label')}

    for outbreak_idx in range(n_countries):
        label = labels[outbreak_idx]
        outbreak_country = source_countries[outbreak_idx]

        # Extract cumulative cumulants for this country
        cum_total = grp_total[label]
        cum_latent = grp_latent[label]
        cum_infectious = grp_infectious[label]

        # Vectorized cumulative-to-daily 
        daily_total = np.empty_like(cum_total)
        daily_total[:-1] = np.diff(cum_total)
        daily_total[-1] = cum_total[-1]

        daily_latent = np.empty_like(cum_latent)
        daily_latent[:-1] = np.diff(cum_latent)
        daily_latent[-1] = cum_latent[-1]

        daily_infectious = np.empty_like(cum_infectious)
        daily_infectious[:-1] = np.diff(cum_infectious)
        daily_infectious[-1] = cum_infectious[-1]

        # Detectable (I+P) = Total (L+I+P) - Latent (L), via cumulant additivity
        daily_detectable = daily_total - daily_latent

        # Detection time for this country
        mean_det_time = det_time_map.get(label, float(T_max))

        # Build chunk DataFrame
        chunk = pd.DataFrame({
            'outbreak_country': outbreak_country,
            'import_country': source_countries[import_country],
            'time': time_arr,
            'daily_total_imports': daily_total[:T_max].astype(float),
            'daily_latent_imports': daily_latent[:T_max].astype(float),
            'daily_infectious_imports': daily_infectious[:T_max].astype(float),
            'daily_detectable_imports': daily_detectable[:T_max].astype(float),
            'mean_detection_time': float(mean_det_time),
            'R0': float(R0),
            'generation_time': float(gen_time)
        })
        chunks.append(chunk)

    print(f"  Done — {n_countries} countries processed for import_country: {source_countries[import_country]}")

# ============================================================
# Concatenate and save
# ============================================================
daily_df = pd.concat(chunks, ignore_index=True)
daily_df.to_csv(output_filepath, index=False)
print(f"\nComplete! Saved {len(daily_df)} rows to {output_filepath}")


In [ ]:
# daily_imports_with_gamma.csv is a csv for one specific OL; 
# it has the full gamma fit of the importation distributions by the country origin

# This is a legacy version and is no longer used - but a possible extension

# Define output filepath
output_filepath = 'daily_imports_with_gamma.csv'

everything_arr = []

# Track progress
total_combinations = len(source_countries)
completed_combinations = 0

# Iterate over all parameter combinations
for outbreak_idx in range(len(source_countries)):
    outbreak_country = source_countries[outbreak_idx]

    print(f"\nProcessing: {outbreak_country}")
    
    # Set up emat for TOTAL imports (all import countries)
    emat_total = np.zeros([len(smat), len(smat)])
    emat_total[:, sentinel_loc] = 1
    emat_total[sentinel_loc, sentinel_loc] = 0

    smat_s = np.zeros([len(mmat), len(mmat)])
    smat_s[:, sentinel_loc] = 0.16  # Dummy value
    smat_s[sentinel_loc, sentinel_loc] = 0

    # Define states
    infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
    idx_infected = [outbreak_idx]
    weight_infected = [1.0]

    # Create PGF for MEAN of TOTAL imports (with cumulant=True)
    K_mean = Sentinel2PGF(umat, mmat, smat_s, emat_total, latency_period, infectious_period,
                    post_infectious_period, nb_infectious_states,
                    nb_post_infectious_states, infection, nb_microsteps, 
                    cmat1=cmat1, cmat2=cmat2, cumulant=True)
        
    K_mean.add_initial_conditions(idx_infected, weight_infected, 
                            label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                            **infectious_state)

    # Transform for cumulative importations (for mean)
    def transform(state_vars, c_):
        state_vars['cumulative importation'] *= c_
    
    transform_dict = {max(t): [transform]}

    # Get mean of total importations
    importations = marginal_cumulants(K_mean, t, transform_dict, radius=10**(-6), resolution=2)
    df = importations[importations['target'] == 1]

    df_daily = df.copy()
    for k in range(len(df_daily) - 1):
        df_daily['cumulant'].iloc[k] = df_daily['cumulant'].iloc[k + 1] - df_daily['cumulant'].iloc[k]
    df_daily['time'] = df_daily['time'].shift(-1)
    
    # Store daily mean imports
    daily_mean_imports = df_daily['cumulant'].values[:121].copy()
    
    # Create PGF for TOTAL cumulative distribution (without cumulant=True)
    K_total = Sentinel2PGF(umat, mmat, smat_s, emat_total, latency_period, infectious_period,
                    post_infectious_period, nb_infectious_states,
                    nb_post_infectious_states, infection, nb_microsteps, 
                    cmat1=cmat1, cmat2=cmat2)  # NO cumulant=True!
    
    K_total.add_initial_conditions(idx_infected, weight_infected, 
                            label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                            **infectious_state)
    
    # Transform for cumulative importations
    def transform_total(state_vars, c_):
        state_vars['cumulative importation'] *= c_
    
    transform_dict_total = {max(t): [transform_total]}
    
    # Get CUMULATIVE distribution for TOTAL imports at each time point
    total_cumulative_distributions = marginal_distribution(K_total, t, transform_dict_total, 
                                                    resolution=500, cutoff=100)
    total_cumulative_distributions['probability'] = np.real(total_cumulative_distributions['probability'])
    
    # Fit Gamma to total cumulative distribution at each time point
    gamma_shape_arr = [np.nan] * T_max  # alpha
    gamma_rate_arr = [np.nan] * T_max   # beta
    
    # Find significant time points for total imports
    significant_times = [i for i in range(min(len(total_mean_imports), T_max)) 
                        if total_mean_imports[i] > 0.1]
    
    
    # Track the first time mean imports exceeds 20 for plotting
    first_time_exceeds_20 = None
    
    for time_point in significant_times:
        time_data = total_cumulative_distributions[total_cumulative_distributions['time'] == time_point]
        
        if len(time_data) == 0:
            continue
        
        for label, df_imp in time_data.groupby('label'):
            target = df_imp['target'].values
            probs = df_imp['probability'].values
            
            # Normalize probabilities
            total_prob = np.sum(probs)
            if total_prob > 0 and np.isfinite(total_prob):
                probs = probs / total_prob
            else:
                continue
            
            # Find mode
            mode_index = np.argmax(probs)
            mode_target = target[mode_index]
            mode_prob = probs[mode_index]
            
            # Fit Gamma to cumulatuive distribution using mode
            # For Gamma: mode = (alpha - 1) / beta (for alpha > 1)
            # PDF at mode = (beta^alpha / Gamma(alpha)) * mode^(alpha-1) * exp(-beta*mode)
            # We have two equations: mode location and PDF height
            
            if mode_target > 0 and mode_prob > 1e-10 and not np.isnan(mode_prob):
                # Use numerical optimization to find alpha and beta
                from scipy.optimize import minimize_scalar
                
                def objective(alpha):
                    if alpha <= 1:
                        return 1e10
                    beta = (alpha - 1) / mode_target
                    fitted_mode = (alpha - 1) / beta
                    fitted_pdf = gamma.pdf(fitted_mode, a=alpha, scale=1/beta)
                    return (fitted_pdf - mode_prob)**2
                
                try:
                    result = minimize_scalar(objective, bounds=(1.1, 100), method='bounded')
                    alpha = result.x
                    beta = (alpha - 1) / mode_target
                    
                    # Verify the fit
                    fitted_mode = (alpha - 1) / beta
                    fitted_pdf = gamma.pdf(fitted_mode, a=alpha, scale=1/beta)
                    
                    if abs(fitted_mode - mode_target) < 0.1 and abs(fitted_pdf - mode_prob) < 0.001:
                        gamma_shape_arr[time_point] = alpha
                        gamma_rate_arr[time_point] = beta
                        
                        if time_point in significant_times[:5]:
                            
                except Exception as e:
                    continue
                
    fitted_count = np.sum(~np.isnan(gamma_shape_arr))
    
    # Set up detection calculation
    smat_s = np.zeros([len(mmat), len(mmat)])
    smat_s[:, sentinel_loc] = p_det
    smat_s[sentinel_loc, sentinel_loc] = 0
    
    K_det = SentinelPGF(umat, mmat, smat_s, latency_period, infectious_period,
                    post_infectious_period, nb_infectious_states,
                    nb_post_infectious_states, infection, nb_microsteps, 
                    cmat1=cmat1, cmat2=cmat2)
        
    infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
    idx_infected = [outbreak_idx]
    weight_infected = [1.0]
    
    K_det.add_initial_conditions(idx_infected, weight_infected, 
                            label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                            **infectious_state)

    def det_transform(state_vars, c_):
        state_vars['cumulative detection'] *= c_
        
    transform_det_dict = {max(t): [det_transform]}

    try:
        detections = marginal_p0(K_det, t, transform_det_dict)
        detections['probability'] = np.real(detections['probability'])

        detection_time = get_time_to_detection(detections, 1, 
                                                quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                                                groupbycols=['label'], 
                                                turnaround_time=turnaround_time)
        
        mean_detection_time = detection_time['mean'].values[0]
        print(f"    Detection time: {mean_detection_time:.2f} days")

    except Exception as e:
        mean_detection_time = T_max

    # Store data for each time point
    for time_idx, time_val in enumerate(t):
        if time_idx <= 120:
            everything_arr.append([
                outbreak_country,
                time_val,
                daily_mean_imports[time_idx],  # TOTAL mean imports
                mean_detection_time,
                gamma_shape_arr[time_idx],
                gamma_rate_arr[time_idx]
            ])
    
    # Save/update CSV after each country
    completed_combinations += 1
    daily_df = pd.DataFrame(everything_arr, columns=['outbreak_country', 'time', 
                                                        'daily_mean_imports', 'mean_detection_time', 
                                                        'gamma_cumulative_shape', 'gamma_cumulative_rate'])
    daily_df.to_csv(output_filepath, index=False)

In [ ]:
# daily_imports_ghana.csv is a csv with ghana (as example) as the OL, and all countries divided as importers; 
# this csv considers the time to first arrival *everywhere* (not just in the UK), and outputs the local infections 
# over time
import warnings
warnings.filterwarnings("ignore")
T_max = 150
t = list(range(0, T_max))
R0 = 2
generation_time = 4
pdet = 0.16 # this doesnt matter as no detection
turnaround_time_imp = 0 # neither does this, set to 0 as not considering detection
# Calculate infectious_period and latency_period from generation time
mean_infectious_period = 8.0 / 3.0  # ~2.67 days
latency_period = generation_time - 0.75 * mean_infectious_period
infectious_period = mean_infectious_period
# Define output filepath
output_filepath = 'daily_imports_ghana.csv'
everything_arr = []
choice_ol = source_countries.index('Ghana')
print(f"Starting simulation for outbreak in {source_countries[choice_ol]}")
print(f"Processing {len(source_countries)} destination countries...")

def find_trim_index(cases_array):
    """
    Find the index where cases return to 0 after initially increasing.
    Returns the index where we should stop (trim from index-2 onwards).
    Works on raw (unrounded) values.
    """
    # Find indices where cases are effectively non-zero (use small threshold for numerical precision)
    non_zero_indices = np.where(cases_array > 1e-10)[0]
    
    if len(non_zero_indices) == 0:
        # All zeros - keep everything
        return len(cases_array)
    
    last_nonzero = non_zero_indices[-1]
    
    # Check if there's a gap (cases go back to 0)
    if last_nonzero < len(cases_array) - 1:
        # Cases went back to 0
        # We want to trim from (last_nonzero + 1) - 2 = last_nonzero - 1
        trim_index = last_nonzero - 1
        return max(trim_index, 0)  # Don't go negative
    else:
        # Cases never went back to 0, keep everything
        return len(cases_array)

for i in range(len(source_countries)):
    emat_spec = np.zeros([len(smat), len(smat)])
    # We're tracking importations into country i (from everywhere)
    emat_spec[:, i] = 1
    emat_spec[i, i] = 0 # don't count coming back
    smat_s = np.zeros([len(mmat), len(mmat)])
    smat_s[:, sentinel_loc] = p_det
    smat_s[sentinel_loc, sentinel_loc] = 0
    # Track local cases in country i over time
    K = Sentinel2PGF(umat, mmat, smat_s, emat_spec, latency_period, infectious_period,
                    post_infectious_period, nb_infectious_states,
                    nb_post_infectious_states, infection, nb_microsteps, 
                    cmat1=cmat1, cmat2=cmat2, cumulant=True)
    # Define states
    infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
    # Set initial conditions for outbreak location
    idx_infected = [choice_ol]
    weight_infected = [1.0]
    K.add_initial_conditions(idx_infected, weight_infected, 
                            label=f'origin: subpopulation {idx_infected[0]+1} = {source_countries[idx_infected[0]]}', 
                            **infectious_state)
    def transform(state_vars, c_):
        # Create a mask: c_ for country i, 1 for all others
        mask = np.ones(len(state_vars['infectious 1']))
        mask[i] = c_
        state_vars['infectious 1'] = state_vars['infectious 1'] * mask
        state_vars['infectious 2'] = state_vars['infectious 2'] * mask
    
    transform_cases_dict = {max(t): [transform]}
    # Get local infections over time
    local_infectives = marginal_cumulants(K, t, transform_cases_dict, radius=10**(-6), resolution=2)
    df = local_infectives[local_infectives['target'] == 1]
    local_cases_arr = df['cumulant'].values  # This is the time series of local cases (RAW)
    
    # Find where to trim BEFORE rounding (based on raw values)
    trim_index = find_trim_index(local_cases_arr)
    
    # Trim the raw array first
    local_cases_arr_trimmed = local_cases_arr[:trim_index]
    
    # THEN round the trimmed array
    local_cases_arr_trimmed = np.round(local_cases_arr_trimmed).astype(int)
    
    # Expand the TRIMMED time series into long format for CSV storage
    for time_idx in range(len(local_cases_arr_trimmed)):
        everything_arr.append([
            source_countries[choice_ol], 
            source_countries[i],
            time_idx,  # time point
            local_cases_arr_trimmed[time_idx]  # cases at this time (rounded to integer)
        ])
    
    # Print progress
    print(f"Completed {i+1}/{len(source_countries)}: {source_countries[i]} "
          f"(kept {len(local_cases_arr_trimmed)}/{len(local_cases_arr)} time points)")

# Save to CSV in long format
print(f"\nSaving results to {output_filepath}...")
daily_df = pd.DataFrame(everything_arr, columns=['outbreak_country', 'local_country', 'time', 'local_cases'])
daily_df.to_csv(output_filepath, index=False)
print(f"Complete! Saved {len(daily_df)} rows to {output_filepath}")

In [ ]:
# daily_imports_sensitivity_T3.csv is a csv for all countries as OLs - but summed importations; 
# it also does not have the gamma fit parameters, but it iterates over a range of 
# epi and AWW parameters. It also considers only the top-3 import countries for the T3 scenario, 
# and compares to the full set of import countries (benchmark).

# Define the parameter combinations to iterate over
R0_values = [1.5, 2.0, 2.5, 3.0]
generation_time_values = [4, 6, 8, 10]
p_det = 0.16

T_max = 150
t = list(range(0, T_max))

turnaround_time = 3

# Define output filepath
output_filepath = 'daily_imports_sensitivity_T3.csv'

everything_arr = []

# Use baseline parameters for top-3 selection
baseline_gen_time = 8
mean_infectious_period = 8.0 / 3.0
baseline_latency = baseline_gen_time - 0.75 * mean_infectious_period
baseline_infectious = mean_infectious_period

top3_dict = {}  # Will store {outbreak_idx: [list of top 3 import country indices]}

# -------------------------
# FIRST: Identify top-3 import countries for each outbreak
# -------------------------
print("\n=== PHASE 1: Identifying top-3 import countries for each outbreak origin ===\n")

for outbreak_idx in range(len(source_countries)):
    outbreak_country = source_countries[outbreak_idx]
    
    det_means = np.full(len(source_countries), np.inf)
    idx_infected_origin = [outbreak_idx]
    weight_infected_origin = [1.0]
    infectious_state = {'nb_infectious': 10, 'nb_latent': 10}

    for j in range(len(source_countries)):
        try:
            # Build emat_ connecting country j -> sentinel
            emat_ = np.zeros([len(source_countries), len(source_countries)])
            emat_[j, sentinel_loc] = 1
            emat_[sentinel_loc, sentinel_loc] = 0

            smat = np.zeros([len(mmat), len(mmat)])
            smat[:, sentinel_loc] = 0.16
            smat[sentinel_loc, sentinel_loc] = 0.0

            umat_baseline = np.eye(len(normalized_flow_matrix)) * 2.0 / baseline_infectious

            # Use Sentinel3PGF to constrain detection path through country j
            Psi = Sentinel3PGF(umat_baseline, mmat, smat, emat_, baseline_latency, baseline_infectious,
                               post_infectious_period, nb_infectious_states,
                               nb_post_infectious_states, infection, nb_microsteps,
                               cmat1=cmat1, cmat2=cmat2)

            Psi.add_initial_conditions(idx_infected_origin, weight_infected_origin,
                                       label=f'origin: {outbreak_country} -> via {source_countries[j]}',
                                       **infectious_state)

            def det_transform(state_vars, c_):
                state_vars['cumulative detection'] *= c_
            transform_det_dict = {max(t): [det_transform]}

            detections = marginal_p0(Psi, t, transform_det_dict)
            detections['probability'] = np.real(detections['probability'])

            detection_time = get_time_to_detection(detections, D=1,
                                                  quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                                                  groupbycols=['label'],
                                                  turnaround_time=turnaround_time)
            det_means[j] = detection_time['mean'].values[0]

        except Exception as e:
            det_means[j] = np.inf

    # Select top-3 countries (lowest mean detection time)
    top_n = min(3, np.isfinite(det_means).sum())
    if top_n == 0:
        print(f"  Warning: no valid detection computations for {outbreak_country}; using empty list.")
        top3_dict[outbreak_idx] = np.array([], dtype=int)
    else:
        top3_idx = np.argsort(det_means)[:top_n]
        top3_dict[outbreak_idx] = top3_idx
        top3_names = [source_countries[i] for i in top3_idx]
        print(f"  {outbreak_country}: Top {top_n} = {top3_names}")

# Track progress
total_combinations = len(source_countries) * len(R0_values) * len(generation_time_values)
completed_combinations = 0

infectious_period_fix = 8.0 / 3.0  # ~2.67 days

# -------------------------
# NOW ITERATE OVER PARAMETER COMBINATIONS
# -------------------------
print("\n=== PHASE 2: Computing imports and detection times across parameter space ===\n")

for R0 in R0_values:
    for generation_time in generation_time_values:
        # Calculate infectious_period and latency_period from generation time
        latency_period = generation_time - 0.75 * infectious_period_fix
        infectious_period = infectious_period_fix
        
        if latency_period < 0:
            print(f"Skipping: R0={R0}, gen_time={generation_time} (negative latency)")
            continue
        
        print(f"\n{'='*80}")
        print(f"PARAMETER SET: R0={R0}, generation_time={generation_time}, p_det={p_det}")
        print(f"  latency_period={latency_period:.2f}, infectious_period={infectious_period:.2f}")
        print(f"{'='*80}")

        umat_s = np.eye(len(normalized_flow_matrix)) * R0 / infectious_period

        for outbreak_idx in range(len(source_countries)):
            outbreak_country = source_countries[outbreak_idx]

            # Get pre-computed top 3 for this outbreak
            top3_idx = top3_dict[outbreak_idx]
            top3_names = [source_countries[i] for i in top3_idx]
            
            print(f"\nProcessing: {outbreak_country} (top-3: {top3_names})")

            infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
            idx_infected = [outbreak_idx]
            weight_infected = [1.0]

            # -------------------------
            # Build imports for TOP 3 countries (L, I, and I+P)
            # -------------------------
            daily_latent_imports_t3 = np.zeros(len(t))
            daily_infectious_imports_t3 = np.zeros(len(t))
            daily_detectable_imports_t3 = np.zeros(len(t))

            for t3 in top3_idx:
                # Set up emat for this specific import country
                emat_t3 = np.zeros([len(mmat), len(mmat)])
                emat_t3[t3, sentinel_loc] = 1
                emat_t3[sentinel_loc, sentinel_loc] = 0

                smat_s = np.zeros([len(mmat), len(mmat)])
                smat_s[:, sentinel_loc] = p_det
                smat_s[sentinel_loc, sentinel_loc] = 0.0

                K_t3 = Sentinel4PGF(umat_s, mmat, smat_s, emat_t3, 
                                   latency_period, infectious_period,
                                   post_infectious_period, nb_infectious_states,
                                   nb_post_infectious_states, infection, nb_microsteps, 
                                   cmat1=cmat1, cmat2=cmat2, cumulant=True)
                
                K_t3.add_initial_conditions(idx_infected, weight_infected,
                                           label=f'origin: {outbreak_country} -> {source_countries[t3]}',
                                           **infectious_state)
                
                # Transform latent compartment (L)
                def transform_latent(state_vars, c_):
                    state_vars['cumulative_latent_importation'] *= c_
                
                transform_dict_latent = {max(t): [transform_latent]}

                try:
                    importations_latent = marginal_cumulants(K_t3, t, transform_dict_latent, 
                                                           radius=10**(-6), resolution=2)
                    df_latent = importations_latent[importations_latent['target'] == 1]
                    df_daily_latent = df_latent.copy()
                    for k in range(len(df_daily_latent) - 1):
                        df_daily_latent['cumulant'].iloc[k] = df_daily_latent['cumulant'].iloc[k + 1] - df_daily_latent['cumulant'].iloc[k]
                    df_daily_latent['time'] = df_daily_latent['time'].shift(-1)
                    daily_latent_imports_t3 += df_daily_latent['cumulant'].values
                except Exception as e:
                    print(f"    Warning: latent cumulants failed for import-country {source_countries[t3]}: {e}")
                
                # Transform infectious compartment (I)
                def transform_infectious(state_vars, c_):
                    state_vars['cumulative_infectious_importation'] *= c_
                
                transform_dict_infectious = {max(t): [transform_infectious]}

                try:
                    importations_infectious = marginal_cumulants(K_t3, t, transform_dict_infectious, 
                                                               radius=10**(-6), resolution=2)
                    df_infectious = importations_infectious[importations_infectious['target'] == 1]
                    df_daily_infectious = df_infectious.copy()
                    for k in range(len(df_daily_infectious) - 1):
                        df_daily_infectious['cumulant'].iloc[k] = df_daily_infectious['cumulant'].iloc[k + 1] - df_daily_infectious['cumulant'].iloc[k]
                    df_daily_infectious['time'] = df_daily_infectious['time'].shift(-1)
                    daily_infectious_imports_t3 += df_daily_infectious['cumulant'].values
                except Exception as e:
                    print(f"    Warning: infectious cumulants failed for import-country {source_countries[t3]}: {e}")
                
                # Transform detectable compartments (I + P)
                def transform_detectable(state_vars, c_):
                    state_vars['cumulative_infectious_importation'] *= c_
                    state_vars['cumulative_post_infectious_importation'] *= c_
                
                transform_dict = {max(t): [transform_detectable]}

                try:
                    importations = marginal_cumulants(K_t3, t, transform_dict, 
                                                     radius=10**(-6), resolution=2)
                    df = importations[importations['target'] == 1]
                    
                    # Convert cumulative to daily
                    df_daily = df.copy()
                    for k in range(len(df_daily) - 1):
                        df_daily['cumulant'].iloc[k] = df_daily['cumulant'].iloc[k + 1] - df_daily['cumulant'].iloc[k]
                    df_daily['time'] = df_daily['time'].shift(-1)
                    
                    # Accumulate this country's imports
                    daily_detectable_imports_t3 += df_daily['cumulant'].values

                except Exception as e:
                    print(f"    Warning: detectable cumulants failed for import-country {source_countries[t3]}: {e}")

            # -------------------------
            # Build imports for ALL countries (benchmark - L, I, and I+P)
            # -------------------------
            emat_full = np.zeros([len(mmat), len(mmat)])
            emat_full[:, sentinel_loc] = 1
            emat_full[sentinel_loc, sentinel_loc] = 0
            
            smat_full = np.zeros([len(mmat), len(mmat)])
            smat_full[:, sentinel_loc] = p_det
            smat_full[sentinel_loc, sentinel_loc] = 0.0
            
            K_full = Sentinel4PGF(umat_s, mmat, smat_full, emat_full, 
                                 latency_period, infectious_period,
                                 post_infectious_period, nb_infectious_states,
                                 nb_post_infectious_states, infection, nb_microsteps, 
                                 cmat1=cmat1, cmat2=cmat2, cumulant=True)

            K_full.add_initial_conditions(idx_infected, weight_infected,
                                         label=f'origin: {outbreak_country}',
                                         **infectious_state)

            # Transform latent compartment (L)
            def transform_latent_full(state_vars, c_):
                state_vars['cumulative_latent_importation'] *= c_
            
            transform_dict_latent_full = {max(t): [transform_latent_full]}

            try:
                importations_latent_full = marginal_cumulants(K_full, t, transform_dict_latent_full, 
                                                             radius=10**(-6), resolution=2)
                df_latent_full = importations_latent_full[importations_latent_full['target'] == 1]
                df_daily_latent_full = df_latent_full.copy()
                for k in range(len(df_daily_latent_full) - 1):
                    df_daily_latent_full['cumulant'].iloc[k] = df_daily_latent_full['cumulant'].iloc[k + 1] - df_daily_latent_full['cumulant'].iloc[k]
                df_daily_latent_full['time'] = df_daily_latent_full['time'].shift(-1)
                daily_latent_imports_all = df_daily_latent_full['cumulant'].values
            except Exception as e:
                print(f"    Warning: full latent cumulants failed: {e}")
                daily_latent_imports_all = np.zeros(len(t))

            # Transform infectious compartment (I)
            def transform_infectious_full(state_vars, c_):
                state_vars['cumulative_infectious_importation'] *= c_
            
            transform_dict_infectious_full = {max(t): [transform_infectious_full]}

            try:
                importations_infectious_full = marginal_cumulants(K_full, t, transform_dict_infectious_full, 
                                                                 radius=10**(-6), resolution=2)
                df_infectious_full = importations_infectious_full[importations_infectious_full['target'] == 1]
                df_daily_infectious_full = df_infectious_full.copy()
                for k in range(len(df_daily_infectious_full) - 1):
                    df_daily_infectious_full['cumulant'].iloc[k] = df_daily_infectious_full['cumulant'].iloc[k + 1] - df_daily_infectious_full['cumulant'].iloc[k]
                df_daily_infectious_full['time'] = df_daily_infectious_full['time'].shift(-1)
                daily_infectious_imports_all = df_daily_infectious_full['cumulant'].values
            except Exception as e:
                print(f"    Warning: full infectious cumulants failed: {e}")
                daily_infectious_imports_all = np.zeros(len(t))

            # Transform detectable compartments (I + P)
            def transform_detectable_full(state_vars, c_):
                state_vars['cumulative_infectious_importation'] *= c_
                state_vars['cumulative_post_infectious_importation'] *= c_
            
            transform_dict_full = {max(t): [transform_detectable_full]}

            try:
                importations_full = marginal_cumulants(K_full, t, transform_dict_full, 
                                                      radius=10**(-6), resolution=2)
                df_full = importations_full[importations_full['target'] == 1]

                # Convert cumulative to daily
                df_full_daily = df_full.copy()
                for k in range(len(df_full_daily) - 1):
                    df_full_daily['cumulant'].iloc[k] = df_full_daily['cumulant'].iloc[k + 1] - df_full_daily['cumulant'].iloc[k]
                df_full_daily['time'] = df_full_daily['time'].shift(-1)
                
                daily_detectable_imports_all = df_full_daily['cumulant'].values

            except Exception as e:
                print(f"    Warning: full detectable cumulants failed: {e}")
                daily_detectable_imports_all = np.zeros(len(t))

            # -------------------------
            # DETECTION CALCULATION (using top-3 smat)
            # -------------------------
            # Build smat where ONLY top-3 countries can detect
            smat_t3_det = np.zeros_like(mmat)
            for t3 in top3_idx:
                smat_t3_det[t3, sentinel_loc] = p_det
            smat_t3_det[sentinel_loc, sentinel_loc] = 0.0
            
            K_det = SentinelPGF(umat_s, mmat, smat_t3_det, latency_period, infectious_period,
                               post_infectious_period, nb_infectious_states,
                               nb_post_infectious_states, infection, nb_microsteps,
                               cmat1=cmat1, cmat2=cmat2)
            
            K_det.add_initial_conditions(idx_infected, weight_infected,
                                        label=f'origin: {outbreak_country}',
                                        **infectious_state)

            def det_transform(state_vars, c_):
                state_vars['cumulative detection'] *= c_
            transform_det_dict = {max(t): [det_transform]}

            try:
                detections = marginal_p0(K_det, t, transform_det_dict)
                detections['probability'] = np.real(detections['probability'])

                detection_time = get_time_to_detection(detections, D=1,
                                                       quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                                                       groupbycols=['label'],
                                                       turnaround_time=turnaround_time)
                
                mean_detection_time_t3 = detection_time['mean'].values[0]
                print(f"    T3 detection time: {mean_detection_time_t3:.2f} days")

            except Exception as e:
                mean_detection_time_t3 = T_max
                print(f"    T3 detection failed: {e}")
            
            # Detection for ALL countries (benchmark)
            smat_all_det = np.zeros_like(mmat)
            smat_all_det[:, sentinel_loc] = p_det
            smat_all_det[sentinel_loc, sentinel_loc] = 0.0
            
            K_det_all = SentinelPGF(umat_s, mmat, smat_all_det, latency_period, infectious_period,
                                   post_infectious_period, nb_infectious_states,
                                   nb_post_infectious_states, infection, nb_microsteps,
                                   cmat1=cmat1, cmat2=cmat2)
            
            K_det_all.add_initial_conditions(idx_infected, weight_infected,
                                            label=f'origin: {outbreak_country}',
                                            **infectious_state)

            try:
                detections_all = marginal_p0(K_det_all, t, transform_det_dict)
                detections_all['probability'] = np.real(detections_all['probability'])

                detection_time_all = get_time_to_detection(detections_all, D=1,
                                                           quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                                                           groupbycols=['label'],
                                                           turnaround_time=turnaround_time)
                
                mean_detection_time_all = detection_time_all['mean'].values[0]
                print(f"    ALL detection time: {mean_detection_time_all:.2f} days")

            except Exception as e:
                mean_detection_time_all = T_max
                print(f"    ALL detection failed: {e}")

            # Store results: one row per time step
            for time_idx, time_val in enumerate(t):
                if time_idx <= 120:
                    everything_arr.append([
                        R0,
                        generation_time,
                        outbreak_country,
                        time_val,
                        daily_latent_imports_t3[time_idx],      # L imports from top-3
                        daily_infectious_imports_t3[time_idx],  # I imports from top-3
                        daily_detectable_imports_t3[time_idx],  # I+P imports from top-3
                        daily_latent_imports_all[time_idx],     # L imports from all
                        daily_infectious_imports_all[time_idx], # I imports from all
                        daily_detectable_imports_all[time_idx], # I+P imports from all
                        mean_detection_time_t3,                 # detection time with top-3
                        mean_detection_time_all                  # detection time with all countries
                    ])

            # Save/update CSV after each country
            completed_combinations += 1
            daily_df = pd.DataFrame(everything_arr, columns=['R0', 'generation_time', 
                                                            'outbreak_country', 'time', 
                                                            'daily_latent_imports_top3',
                                                            'daily_infectious_imports_top3',
                                                            'daily_detectable_imports_top3',
                                                            'daily_latent_imports_all',
                                                            'daily_infectious_imports_all',
                                                            'daily_detectable_imports_all',
                                                            'mean_detection_time_top3',
                                                            'mean_detection_time_all'])
            daily_df.to_csv(output_filepath, index=False)
            
            print(f"  Progress: {completed_combinations}/{total_combinations} combinations completed")

Gamma test

In [ ]:
print(f"infectious period: {infectious_period}, infectious_period_fix: {infectious_period_fix}, latency_period: {latency_period}")

In [ ]:
# daily_imports_specific_sensitivity_gamma is a csv for user-defined countries as OLs; 
# it has the gamma fit parameters and iterates over a range of epi and AWW parameters


# Define the parameter combinations to iterate over
selected_countries = ['Switzerland', 'South Korea', 'Paraguay']
R0_values = [1.5, 2.0, 2.5, 3.0]
generation_times = [4, 6, 8, 10]
p_det_values = [0.04, 0.08, 0.16]

T_max = 200
t = list(range(0, T_max))

# Define output filepath
output_filepath = 'daily_imports_specific_sensitivity_gamma.csv'

# Get indices of selected countries
selected_indices = [i for i, country in enumerate(source_countries) if country in selected_countries]

everything_arr = []

# Track progress
total_combinations = len(selected_countries) * len(R0_values) * len(generation_times) * len(p_det_values)
completed_combinations = 0

infectious_period_fix = 8.0 / 3.0  # ~2.67 days

# Iterate over all parameter combinations
for outbreak_idx in selected_indices:
    outbreak_country = source_countries[outbreak_idx]
    
    for R0 in R0_values:
        for gen_time in generation_times:
            
            print(f"\nProcessing: {outbreak_country}, R0={R0}, gen_time={gen_time}")
            
            latency_period = gen_time - 0.75 * infectious_period_fix
            infectious_period = infectious_period_fix

            # make sure positive latency period
            if latency_period < 0:
                continue

    
            smat_s = np.zeros([len(mmat), len(mmat)])
            smat_s[:, sentinel_loc] = 0.16  # Dummy value
            smat_s[sentinel_loc, sentinel_loc] = 0

            # Set up emat for TOTAL imports (all import countries)
            emat_total = np.zeros([len(smat), len(smat)])
            emat_total[:, sentinel_loc] = 1
            emat_total[sentinel_loc, sentinel_loc] = 0

            umat_s = np.eye(len(normalized_flow_matrix)) * R0 / mean_infectious_period

            # Create PGF for MEAN of TOTAL imports (with cumulant=True)
            K_mean = Sentinel2PGF(umat_s, mmat, smat_s, emat_total, latency_period, infectious_period,
                            post_infectious_period, nb_infectious_states,
                            nb_post_infectious_states, infection, nb_microsteps, 
                            cmat1=cmat1, cmat2=cmat2, cumulant=True)
                
            # Define states
            infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
            idx_infected = [outbreak_idx]
            weight_infected = [1.0]
            
            K_mean.add_initial_conditions(idx_infected, weight_infected, 
                                    label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                                    **infectious_state)

            # Transform for cumulative importations (for mean)
            def transform(state_vars, c_):
                state_vars['cumulative importation'] *= c_
            
            transform_dict = {max(t): [transform]}

            # Get MEAN of TOTAL importations
            importations = marginal_cumulants(K_mean, t, transform_dict, radius=10**(-6), resolution=2)
            df = importations[importations['target'] == 1]

            df_daily = df.copy()
            for k in range(len(df_daily) - 1):
                df_daily['cumulant'].iloc[k] = df_daily['cumulant'].iloc[k + 1] - df_daily['cumulant'].iloc[k]
            df_daily['time'] = df_daily['time'].shift(-1)
            
            # Store total mean imports
            daily_mean_imports = df_daily['cumulant'].values[:121].copy()
            
            # Create PGF for TOTAL cumulative distribution (without cumulant=True)
            K_total = Sentinel2PGF(umat_s, mmat, smat_s, emat_total, latency_period, infectious_period,
                            post_infectious_period, nb_infectious_states,
                            nb_post_infectious_states, infection, nb_microsteps, 
                            cmat1=cmat1, cmat2=cmat2)  # NO cumulant=True!
            
            K_total.add_initial_conditions(idx_infected, weight_infected, 
                                    label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                                    **infectious_state)
            
            # Transform for cumulative importations
            def transform_total(state_vars, c_):
                state_vars['cumulative importation'] *= c_
            
            transform_dict_total = {max(t): [transform_total]}
            
            # Get cumulative distribution for total imports at each time point
            total_cumulative_distributions = marginal_distribution(K_total, t, transform_dict_total, 
                                                            resolution=500, cutoff=100)
            total_cumulative_distributions['probability'] = np.real(total_cumulative_distributions['probability'])
            
            # Fit Gamma to total cumulative distribution at each time point
            gamma_shape_arr = [np.nan] * T_max  # alpha
            gamma_rate_arr = [np.nan] * T_max   # beta
            
            # Find significant time points for total imports
            significant_times = [i for i in range(min(len(daily_mean_imports), T_max)) 
                                if daily_mean_imports[i] > 0.1]
            
            for time_point in significant_times:
                time_data = total_cumulative_distributions[total_cumulative_distributions['time'] == time_point]
                
                if len(time_data) == 0:
                    continue
                
                for label, df_imp in time_data.groupby('label'):
                    target = df_imp['target'].values
                    probs = df_imp['probability'].values
                    
                    # Normalize probabilities
                    total_prob = np.sum(probs)
                    if total_prob > 0 and np.isfinite(total_prob):
                        probs = probs / total_prob
                    else:
                        continue
                    
                    # Find mode
                    mode_index = np.argmax(probs)
                    mode_target = target[mode_index]
                    mode_prob = probs[mode_index]
                    
                    # Gamma distribution modal fitting: mode = (alpha - 1) / beta (for alpha > 1)
                    # PDF at mode = (beta^alpha / Gamma(alpha)) * mode^(alpha-1) * exp(-beta*mode)
                
                    if mode_target > 0 and mode_prob > 1e-10 and not np.isnan(mode_prob):
                        # Use numerical optimization to find alpha and beta
                        from scipy.optimize import minimize_scalar
                        
                        def objective(alpha):
                            if alpha <= 1:
                                return 1e10
                            beta = (alpha - 1) / mode_target
                            fitted_mode = (alpha - 1) / beta
                            fitted_pdf = gamma.pdf(fitted_mode, a=alpha, scale=1/beta)
                            return (fitted_pdf - mode_prob)**2
                        
                        try:
                            result = minimize_scalar(objective, bounds=(1.1, 100), method='bounded')
                            alpha = result.x
                            beta = (alpha - 1) / mode_target
                            
                            # Verify the fit
                            fitted_mode = (alpha - 1) / beta
                            fitted_pdf = gamma.pdf(fitted_mode, a=alpha, scale=1/beta)
                            
                            if abs(fitted_mode - mode_target) < 0.1 and abs(fitted_pdf - mode_prob) < 0.001:
                                gamma_shape_arr[time_point] = alpha
                                gamma_rate_arr[time_point] = beta
                                
                                if time_point in significant_times[:5]:
                                    print(f"    Fitted Gamma at time {time_point}: α={alpha:.2f}, β={beta:.4f}")
                            
                        except Exception as e:
                            continue
                        
            fitted_count = np.sum(~np.isnan(gamma_shape_arr))
            
            # Now loop over detection probabilities
            for p_det in p_det_values:
            
                # Set up detection calculation
                smat_s = np.zeros([len(mmat), len(mmat)])
                smat_s[:, sentinel_loc] = p_det
                smat_s[sentinel_loc, sentinel_loc] = 0
                
                K_det = SentinelPGF(umat_s, mmat, smat_s, latency_period, infectious_period,
                                post_infectious_period, nb_infectious_states,
                                nb_post_infectious_states, infection, nb_microsteps, 
                                cmat1=cmat1, cmat2=cmat2)
                    
                infectious_state = {'nb_infectious': 10, 'nb_latent': 10}
                idx_infected = [outbreak_idx]
                weight_infected = [1.0]
                
                K_det.add_initial_conditions(idx_infected, weight_infected, 
                                        label=f'origin: subpopulation {idx_infected[0]+1} = {outbreak_country}', 
                                        **infectious_state)

                def det_transform(state_vars, c_):
                    state_vars['cumulative detection'] *= c_
                    
                transform_det_dict = {max(t): [det_transform]}

                try:
                    detections = marginal_p0(K_det, t, transform_det_dict)
                    detections['probability'] = np.real(detections['probability'])

                    detection_time = get_time_to_detection(detections, 1, 
                                                          quantiles=[0.25, 0.5, 0.75, 0.95, 0.99],
                                                          groupbycols=['label'], 
                                                          turnaround_time=turnaround_time)
                    
                    mean_detection_time = detection_time['mean'].values[0]
                    print(f"    Detection time: {mean_detection_time:.2f} days")

                except Exception as e:
                    mean_detection_time = T_max
                    print(f"    Detection failed, using T_max")

                # Store data for each time point
                for time_idx, time_val in enumerate(t):
                    if time_idx <= 120:
                        everything_arr.append([
                            outbreak_country,
                            time_val,
                            daily_mean_imports[time_idx],
                            mean_detection_time,
                            R0,
                            gen_time,
                            p_det,
                            gamma_shape_arr[time_idx],
                            gamma_rate_arr[time_idx]
                        ])
                
                # Save/update CSV after each p_det combination
                completed_combinations += 1
                daily_df = pd.DataFrame(everything_arr, columns=['outbreak_country', 'time', 
                                                                 'daily_mean_imports', 'mean_detection_time', 
                                                                 'R0', 'generation_time', 'p_det',
                                                                 'gamma_cumulative_shape', 'gamma_cumulative_rate'])
                daily_df.to_csv(output_filepath, index=False)
